# Conditional VAE-TimeGAN - Multi-patient parametrisable notebook

This notebook is designed to be executed programmatically by run_all_patients.py
via nbclient, with parameters injected into the cell tagged 'parameters' before
each run. It can also be executed manually by editing the parameters cell directly.

Each run trains one VAE-TimeGAN model for a single patient under a single
configuration (RECOVERY_ACTIVATION). The orchestrator calls this notebook once
per (patient, configuration) combination, with a fresh kernel each time.

## 0. Imports and reproducibility

In [ ]:
import os
import json
import random
import warnings
from pathlib import Path
from typing import Dict, List, Tuple, Optional

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
import matplotlib.pyplot as plt
import matplotlib as mpl
from scipy import stats, signal
from scipy.stats import wasserstein_distance, ks_2samp
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.metrics import roc_auc_score
from sklearn.linear_model import LogisticRegression

warnings.filterwarnings('ignore')

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")
print(f"PyTorch version: {torch.__version__}")


In [ ]:
# ── Parameters cell — overwritten by run_all_patients.py before each execution ──
# When running manually, edit these values directly.
PATIENT_ID_PARAM          = "588"
RECOVERY_ACTIVATION_PARAM = "tanh"          # "tanh" | "none" | "softsign"
RUN_NAME_SUFFIX_PARAM     = "norm"          # used to build RUN_NAME


## 1. Run configuration

In [ ]:
# ── Identification (derived from injected parameters) ──────────────────────────
PATIENT_ID = PATIENT_ID_PARAM
RUN_NAME   = f"vae_wgan_wiener02_{RUN_NAME_SUFFIX_PARAM}_patient{PATIENT_ID_PARAM}"

# ── Paths ──────────────────────────────────────────────────────────────────────
INPUT_FOLDER   = Path("DatasetsProcessed") / f"patient_{PATIENT_ID}"
OUTPUT_BASE    = Path("ResultadosGenerativos") / "TimeGAN_condicional" / f"patient_{PATIENT_ID}" / f"run_{RUN_NAME}"
RAW_FOLDER     = OUTPUT_BASE / "raw"
FIGURES_FOLDER = OUTPUT_BASE / "figuras"
MODEL_FOLDER   = OUTPUT_BASE / "modelo"

# ── Data ───────────────────────────────────────────────────────────────────────
CGM_CHANNEL    = 0
INPUT_CHANNELS = [1, 2, 3]
ALL_CHANNELS   = [0, 1, 2, 3]
WINDOW_LENGTH  = 96

# ── Architecture ───────────────────────────────────────────────────────────────
HIDDEN_DIM  = 32
NUM_LAYERS  = 3
NOISE_DIM   = 64
BETA_KL     = 0.001

# Recovery output activation: injected from parameters cell
RECOVERY_ACTIVATION = RECOVERY_ACTIVATION_PARAM

# Generator input noise
NOISE_TYPE   = "wiener"
WIENER_SIGMA = 0.15

# ── Training ───────────────────────────────────────────────────────────────────
BATCH_SIZE         = 128
LR                 = 1e-3
GAMMA              = 1.0
EPOCHS_AE          = 300
EPOCHS_SUP         = 100
EPOCHS_JOINT       = 400
LAMBDA_GP          = 10
D_STEPS            = 2
INSTANCE_NOISE_STD = 0.05

# ── Generation ─────────────────────────────────────────────────────────────────
N_SYNTHETIC = 500

# ── Setup output folders ───────────────────────────────────────────────────────
if OUTPUT_BASE.exists() and any(OUTPUT_BASE.iterdir()):
    raise FileExistsError(
        f"Folder {OUTPUT_BASE} already exists. "
        f"Delete it manually or change RUN_NAME before re-running."
    )

for d in (OUTPUT_BASE, RAW_FOLDER, FIGURES_FOLDER, MODEL_FOLDER):
    d.mkdir(parents=True, exist_ok=True)

config = {
    "patient_id": PATIENT_ID, "run_name": RUN_NAME,
    "model_type": "wgan_gp", "embedder_type": "vae",
    "seed": SEED, "device": str(DEVICE),
    "window_length": WINDOW_LENGTH, "cgm_channel": CGM_CHANNEL,
    "input_channels": INPUT_CHANNELS,
    "hidden_dim": HIDDEN_DIM, "num_layers": NUM_LAYERS,
    "noise_dim": NOISE_DIM, "beta_kl": BETA_KL,
    "recovery_activation": RECOVERY_ACTIVATION,
    "noise_type": NOISE_TYPE, "wiener_sigma": WIENER_SIGMA,
    "instance_noise_std": INSTANCE_NOISE_STD,
    "batch_size": BATCH_SIZE, "lr": LR, "gamma": GAMMA,
    "epochs_ae": EPOCHS_AE, "epochs_sup": EPOCHS_SUP,
    "epochs_joint": EPOCHS_JOINT, "lambda_gp": LAMBDA_GP,
    "d_steps": D_STEPS, "n_synthetic": N_SYNTHETIC,
}
with open(OUTPUT_BASE / "config.json", "w") as f:
    json.dump(config, f, indent=2)

print("Run configuration:")
for k, v in config.items():
    print(f"  {k}: {v}")


## 2. Data loading

In [ ]:
windows_path  = INPUT_FOLDER / "windows.npy"
metadata_path = INPUT_FOLDER / "metadata.json"

assert windows_path.exists(),  f"Not found: {windows_path}"
assert metadata_path.exists(), f"Not found: {metadata_path}"

data_full = np.load(windows_path).astype(np.float32)
with open(metadata_path) as f:
    meta = json.load(f)

N_orig, L_orig, C = data_full.shape
print(f"Dataset loaded: N={N_orig}, L={L_orig}, C={C}")

if WINDOW_LENGTH > L_orig:
    raise ValueError(f"WINDOW_LENGTH={WINDOW_LENGTH} > L_orig={L_orig}.")
elif WINDOW_LENGTH < L_orig:
    data_full = data_full[:, :WINDOW_LENGTH, :]

L = WINDOW_LENGTH
X_cond = data_full[:, :, INPUT_CHANNELS]
X_cgm  = data_full[:, :, [CGM_CHANNEL]]
X_all  = data_full[:, :, ALL_CHANNELS]
n_input, n_output = X_cond.shape[2], 1

phys = meta["physiological_limits"]

def denorm(x_norm, key):
    """Denormalises from [-1, 1] to original units."""
    lo, hi = phys[key]
    return (x_norm + 1.0) / 2.0 * (hi - lo) + lo

CGM_LO, CGM_HI = phys["cgm"]

X_cond_t = torch.tensor(X_cond, dtype=torch.float32)
X_cgm_t  = torch.tensor(X_cgm,  dtype=torch.float32)
dataset  = TensorDataset(X_cond_t, X_cgm_t)

# Reduce batch size automatically if dataset is smaller than BATCH_SIZE
N_windows = len(dataset)
EFFECTIVE_BATCH_SIZE = BATCH_SIZE
if N_windows < BATCH_SIZE:
    EFFECTIVE_BATCH_SIZE = max(1, N_windows // 2)
    print(f"BATCH_SIZE reduced from {BATCH_SIZE} to {EFFECTIVE_BATCH_SIZE} "
          f"(dataset has {N_windows} windows).")

if N_windows < 4:
    raise ValueError(f"Patient {PATIENT_ID}: only {N_windows} windows — insufficient for training.")

dataloader = DataLoader(dataset, batch_size=EFFECTIVE_BATCH_SIZE, shuffle=True, drop_last=True)

if EFFECTIVE_BATCH_SIZE != BATCH_SIZE:
    config["batch_size_effective"] = EFFECTIVE_BATCH_SIZE
    with open(OUTPUT_BASE / "config.json", "w") as f:
        json.dump(config, f, indent=2)

print(f"DataLoader: {len(dataloader)} batches of size {EFFECTIVE_BATCH_SIZE}")

cgm_real_mgdl = denorm(X_cgm.flatten(), "cgm")
print(f"\nReal CGM (mg/dL): mean={cgm_real_mgdl.mean():.1f}  std={cgm_real_mgdl.std():.1f}  "
      f"TIR={((cgm_real_mgdl >= 70) & (cgm_real_mgdl <= 180)).mean()*100:.1f}%")


## 3. Conditional VAE-TimeGAN architecture

In [ ]:
class GRUNet(nn.Module):
    """Reusable GRU block with linear output projection."""
    def __init__(self, input_dim, hidden_dim, output_dim,
                 num_layers=3, dropout=0.0, activation="tanh"):
        super().__init__()
        self.gru = nn.GRU(
            input_size=input_dim, hidden_size=hidden_dim,
            num_layers=num_layers, batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0,
        )
        self.fc  = nn.Linear(hidden_dim, output_dim)
        self.act = {"tanh": nn.Tanh(), "sigmoid": nn.Sigmoid(),
                    "softsign": nn.Softsign(), "none": nn.Identity()}[activation]

    def forward(self, x):
        out, _ = self.gru(x)
        return self.act(self.fc(out))


class GRUVAEEncoder(nn.Module):
    """GRU-based VAE encoder. Produces μ, log σ² and z via reparametrization trick."""
    def __init__(self, input_dim, hidden_dim, num_layers):
        super().__init__()
        self.gru = nn.GRU(
            input_size=input_dim, hidden_size=hidden_dim,
            num_layers=num_layers, batch_first=True,
            dropout=0.1 if num_layers > 1 else 0.0,
        )
        self.fc_mu     = nn.Linear(hidden_dim, hidden_dim)
        self.fc_logvar = nn.Linear(hidden_dim, hidden_dim)

    def forward(self, x):
        h, _   = self.gru(x)
        mu     = self.fc_mu(h)
        logvar = torch.clamp(self.fc_logvar(h), min=-4.0, max=4.0)
        std    = torch.exp(0.5 * logvar)
        z      = mu + std * torch.randn_like(std)
        return mu, logvar, z

    def kl_loss(self, mu, logvar):
        """KL divergence between N(μ, σ²) and N(0, I) in closed form."""
        return (-0.5 * (1 + logvar - mu.pow(2) - logvar.exp())).mean()


class TimeGANVAE(nn.Module):
    """Conditional TimeGAN with VAE encoder and WGAN-GP adversarial objective."""
    def __init__(self, n_input, n_output, hidden_dim, noise_dim,
                 num_layers, recovery_activation="tanh"):
        super().__init__()
        self.noise_dim = noise_dim
        self.vae_encoder   = GRUVAEEncoder(n_output, hidden_dim, num_layers)
        self.recovery      = GRUNet(hidden_dim, hidden_dim, n_output,
                                    num_layers, activation=recovery_activation)
        self.generator     = GRUNet(noise_dim + n_input, hidden_dim, hidden_dim,
                                    num_layers, activation="tanh")
        self.supervisor    = GRUNet(hidden_dim, hidden_dim, hidden_dim,
                                    max(1, num_layers - 1), activation="tanh")
        self.discriminator = GRUNet(hidden_dim, hidden_dim, 1,
                                    max(1, num_layers - 1), activation="none")

    def encode(self, x):       return self.vae_encoder(x)
    def decode(self, z):       return self.recovery(z)
    def generate(self, z, c):  return self.generator(torch.cat([z, c], dim=-1))
    def supervise(self, h):    return self.supervisor(h)
    def discriminate(self, h): return self.discriminator(h)

    @torch.no_grad()
    def sample(self, c, n=None):
        self.eval()
        if n is not None and c.shape[0] == 1:
            c = c.repeat(n, 1, 1)
        B, L, _ = c.shape
        z_noise = sample_noise(B, L, self.noise_dim, c.device, NOISE_TYPE, WIENER_SIGMA)
        return self.decode(self.generate(z_noise, c))


def sample_noise(batch_size, seq_len, noise_dim, device,
                 noise_type="gaussian", sigma=0.05, normalize_wiener=True):
    """Samples generator input noise (Gaussian i.i.d. or normalised Wiener process)."""
    if noise_type == "gaussian":
        return torch.randn(batch_size, seq_len, noise_dim, device=device)
    elif noise_type == "wiener":
        increments = torch.randn(batch_size, seq_len, noise_dim, device=device) * sigma
        wiener = torch.cumsum(increments, dim=1)
        if normalize_wiener:
            t = torch.arange(1, seq_len + 1, device=device, dtype=wiener.dtype)
            wiener = wiener / (sigma * torch.sqrt(t).view(1, seq_len, 1))
        return wiener
    else:
        raise ValueError(f"Unknown noise_type: {noise_type}")


model = TimeGANVAE(
    n_input=n_input, n_output=n_output,
    hidden_dim=HIDDEN_DIM, noise_dim=NOISE_DIM,
    num_layers=NUM_LAYERS, recovery_activation=RECOVERY_ACTIVATION,
).to(DEVICE)

def count_params(m): return sum(p.numel() for p in m.parameters() if p.requires_grad)
print(f"Model initialised. Total params: {count_params(model):,}")
print(f"Patient: {PATIENT_ID} | Recovery activation: {RECOVERY_ACTIVATION} | Noise: {NOISE_TYPE}")


## 4. Optimisers and loss functions

In [ ]:
opt_ae   = torch.optim.Adam(list(model.vae_encoder.parameters()) + list(model.recovery.parameters()), lr=LR)
opt_sup  = torch.optim.Adam(model.supervisor.parameters(), lr=LR)
opt_gen  = torch.optim.Adam(list(model.generator.parameters()) + list(model.supervisor.parameters()), lr=LR)
opt_disc = torch.optim.Adam(model.discriminator.parameters(), lr=LR)

mse_loss = nn.MSELoss()

def gradient_penalty(h_real, h_fake):
    """WGAN-GP gradient penalty."""
    B, L, H = h_real.shape
    alpha = torch.rand(B, 1, 1, device=h_real.device)
    h_hat = (alpha * h_real + (1 - alpha) * h_fake).requires_grad_(True)
    d_hat = model.discriminate(h_hat)
    grad  = torch.autograd.grad(
        outputs=d_hat, inputs=h_hat,
        grad_outputs=torch.ones_like(d_hat),
        create_graph=True, retain_graph=True, only_inputs=True
    )[0]
    return ((grad.norm(2, dim=-1) - 1) ** 2).mean()

def disc_loss(d_real, d_fake, h_real=None, h_fake=None):
    return -d_real.mean() + d_fake.mean() + LAMBDA_GP * gradient_penalty(h_real, h_fake)

def gen_adv_loss(d_fake):
    return -d_fake.mean()

history = {"epoch": [], "phase": [], "loss_ae_recon": [],
           "loss_sup": [], "loss_gen_adv": [], "loss_gen_sup": [], "loss_disc": []}

def log_loss(epoch, phase, **kwargs):
    history["epoch"].append(epoch)
    history["phase"].append(phase)
    for key in ["loss_ae_recon", "loss_sup", "loss_gen_adv", "loss_gen_sup", "loss_disc"]:
        history[key].append(kwargs.get(key, float("nan")))


## 5. Training - Phase 1: VAE (Encoder + Recovery)

In [ ]:
print(f"{'='*60}\n Phase 1: VAE ({EPOCHS_AE} epochs)\n{'='*60}")
model.train()
LOG_INTERVAL  = max(1, EPOCHS_AE // 10)
ANNEAL_EPOCHS = min(50, EPOCHS_AE // 4)

for epoch in range(1, EPOCHS_AE + 1):
    epoch_recon, epoch_kl = 0.0, 0.0
    beta_current = BETA_KL * min(1.0, epoch / ANNEAL_EPOCHS)

    for batch_cond, batch_cgm in dataloader:
        batch_cgm = batch_cgm.to(DEVICE)
        mu, logvar, z = model.encode(batch_cgm)
        l_recon = mse_loss(model.decode(z), batch_cgm)
        l_kl    = model.vae_encoder.kl_loss(mu, logvar)
        loss_ae = l_recon + beta_current * l_kl
        opt_ae.zero_grad()
        loss_ae.backward()
        torch.nn.utils.clip_grad_norm_(
            list(model.vae_encoder.parameters()) + list(model.recovery.parameters()), max_norm=5.0)
        opt_ae.step()
        epoch_recon += l_recon.item()
        epoch_kl    += l_kl.item()

    avg_recon = epoch_recon / len(dataloader)
    avg_kl    = epoch_kl    / len(dataloader)
    log_loss(epoch, "ae", loss_ae_recon=avg_recon)
    if epoch % LOG_INTERVAL == 0 or epoch == 1:
        print(f"  Epoch {epoch:4d}/{EPOCHS_AE} | Recon={avg_recon:.5f}  KL={avg_kl:.5f}  β={beta_current:.5f}")

print(f"\nPhase 1 complete. Recon={avg_recon:.5f}  KL={avg_kl:.5f}")


## 6. Training - Phase 2: Supervisor

In [ ]:
print(f"{'='*60}\n Phase 2: Supervisor ({EPOCHS_SUP} epochs)\n{'='*60}")
model.train()
LOG_INTERVAL = max(1, EPOCHS_SUP // 10)

for epoch in range(1, EPOCHS_SUP + 1):
    epoch_loss = 0.0
    for batch_cond, batch_cgm in dataloader:
        batch_cgm = batch_cgm.to(DEVICE)
        with torch.no_grad():
            _, _, z = model.encode(batch_cgm)
        z_sup    = model.supervise(z[:, :-1, :])
        loss_sup = mse_loss(z_sup, z[:, 1:, :])
        opt_sup.zero_grad()
        loss_sup.backward()
        torch.nn.utils.clip_grad_norm_(model.supervisor.parameters(), max_norm=5.0)
        opt_sup.step()
        epoch_loss += loss_sup.item()

    avg_loss = epoch_loss / len(dataloader)
    log_loss(epoch, "sup", loss_sup=avg_loss)
    if epoch % LOG_INTERVAL == 0 or epoch == 1:
        print(f"  Epoch {epoch:4d}/{EPOCHS_SUP} | SUP loss: {avg_loss:.6f}")

print(f"\nPhase 2 complete. Final SUP loss: {avg_loss:.6f}")


## 7. Training - Phase 3: Joint adversarial training

In [ ]:
print(f"{'='*60}\n Phase 3: Joint adversarial ({EPOCHS_JOINT} epochs) — WGAN-GP\n{'='*60}")
model.train()
LOG_INTERVAL = max(1, EPOCHS_JOINT // 20)

for epoch in range(1, EPOCHS_JOINT + 1):
    g_losses_adv, g_losses_sup, d_losses = [], [], []
    noise_std = INSTANCE_NOISE_STD * (1.0 - (epoch - 1) / EPOCHS_JOINT)

    for batch_cond, batch_cgm in dataloader:
        batch_cond = batch_cond.to(DEVICE)
        batch_cgm  = batch_cgm.to(DEVICE)
        B = batch_cgm.shape[0]

        # Discriminator step
        for _ in range(D_STEPS):
            z_noise = sample_noise(B, L, NOISE_DIM, DEVICE, NOISE_TYPE, WIENER_SIGMA)
            mu, logvar, h_real = model.encode(batch_cgm)
            h_fake = model.generate(z_noise, batch_cond)
            if noise_std > 1e-6:
                h_real_in = h_real + torch.randn_like(h_real) * noise_std
                h_fake_in = h_fake + torch.randn_like(h_fake) * noise_std
            else:
                h_real_in, h_fake_in = h_real, h_fake
            loss_d = disc_loss(
                model.discriminate(h_real_in),
                model.discriminate(h_fake_in.detach()),
                h_real.detach(), h_fake.detach()
            )
            opt_disc.zero_grad()
            loss_d.backward()
            torch.nn.utils.clip_grad_norm_(model.discriminator.parameters(), max_norm=5.0)
            opt_disc.step()
            d_losses.append(loss_d.item())

        # Generator step
        z_noise = sample_noise(B, L, NOISE_DIM, DEVICE, NOISE_TYPE, WIENER_SIGMA)
        h_fake  = model.generate(z_noise, batch_cond)
        l_adv   = gen_adv_loss(model.discriminate(h_fake))
        l_sup   = mse_loss(model.supervise(h_fake[:, :-1, :]), h_fake[:, 1:, :].detach())
        mu, logvar, h_real = model.encode(batch_cgm)
        l_recon = mse_loss(model.decode(h_real), batch_cgm)
        l_kl    = model.vae_encoder.kl_loss(mu, logvar)
        loss_g  = l_adv + GAMMA * l_sup + l_recon + BETA_KL * l_kl

        opt_gen.zero_grad()
        opt_ae.zero_grad()
        loss_g.backward()
        torch.nn.utils.clip_grad_norm_(
            list(model.generator.parameters()) + list(model.supervisor.parameters()), max_norm=5.0)
        opt_gen.step()
        opt_ae.step()
        g_losses_adv.append(l_adv.item())
        g_losses_sup.append(l_sup.item())

    avg_g_adv = np.mean(g_losses_adv)
    avg_g_sup = np.mean(g_losses_sup)
    avg_d     = np.mean(d_losses)
    log_loss(epoch, "joint", loss_gen_adv=avg_g_adv, loss_gen_sup=avg_g_sup, loss_disc=avg_d)
    if epoch % LOG_INTERVAL == 0 or epoch == 1:
        print(f"  Epoch {epoch:4d}/{EPOCHS_JOINT} | G_adv={avg_g_adv:.4f}  G_sup={avg_g_sup:.4f}  D={avg_d:.4f}")

print(f"\nPhase 3 complete.")

for name, net in [("vae_encoder", model.vae_encoder), ("recovery", model.recovery),
                   ("generator", model.generator), ("supervisor", model.supervisor),
                   ("discriminator", model.discriminator)]:
    torch.save(net.state_dict(), MODEL_FOLDER / f"{name}.pt")
print(f"Weights saved to {MODEL_FOLDER}")

pd.DataFrame(history).to_csv(RAW_FOLDER / "losses.csv", index=False)
print(f"Loss history saved to {RAW_FOLDER / 'losses.csv'}")


## 8. Synthetic sample generation and metric computation

In [ ]:
model.eval()
idx_cond = np.random.choice(len(X_cond), size=N_SYNTHETIC, replace=True)
c_sample = torch.tensor(X_cond[idx_cond], dtype=torch.float32).to(DEVICE)

with torch.no_grad():
    cgm_fake_norm = model.sample(c_sample).cpu().numpy()

pct_clipped   = (np.abs(cgm_fake_norm) > 1.0).mean() * 100
cgm_fake_norm = np.clip(cgm_fake_norm, -1.0, 1.0)

cgm_synt_mgdl = denorm(cgm_fake_norm.flatten(), "cgm")
cgm_real_mgdl = denorm(X_cgm.flatten(), "cgm")

print(f"Synthetic: {cgm_fake_norm.shape}  ({pct_clipped:.3f}% values clipped)")
print(f"Synthetic CGM: mean={cgm_synt_mgdl.mean():.1f}  std={cgm_synt_mgdl.std():.1f} mg/dL")

np.save(RAW_FOLDER / "marginal_real.npy", cgm_real_mgdl)
np.save(RAW_FOLDER / "marginal_synt.npy", cgm_synt_mgdl)
np.save(RAW_FOLDER / "windows_synt_mgdl.npy",
        denorm(cgm_fake_norm, "cgm").reshape(N_SYNTHETIC, L, 1))
np.save(RAW_FOLDER / "windows_real_mgdl.npy", denorm(X_cgm, "cgm"))
